# 04 · Function (tool) calling

The model proposes a tool call with JSON arguments. Your code validates the arguments,
runs the function, and returns the result. **The model never executes code itself.**

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from tools import Tool, ToolRegistry, get_weather, safe_eval, run_tool_loop
from llm_utils import get_llm

registry = ToolRegistry()
registry.register(Tool(
    name="get_weather",
    description="Get the current weather forecast for a city.",
    parameters={"type": "object",
                "properties": {"city": {"type": "string", "description": "City name"}},
                "required": ["city"]},
    fn=get_weather,
))
registry.register(Tool(
    name="calculator",
    description="Evaluate an arithmetic expression using + - * / and parentheses.",
    parameters={"type": "object",
                "properties": {"expression": {"type": "string"}},
                "required": ["expression"]},
    fn=lambda expression: {"result": safe_eval(expression)},
))
print([s["function"]["name"] for s in registry.schemas()])

## 1. Validation happens before anything runs

In [ ]:
print(registry.dispatch("get_weather", '{"city": "Paris"}'))
print(registry.dispatch("get_weather", '{}'))
print(registry.dispatch("get_weather", '{"city": "Paris", "units": "C"}'))
print(registry.dispatch("calculator", '{"expression": "10 / 0"}'))
print(registry.dispatch("delete_files", '{}'))
print(registry.dispatch("calculator", 'not json'))

## 2. Safe expression evaluation (no `eval`)

In [ ]:
print(safe_eval("2 * (3 + 4) - 5 / 2"))
try:
    safe_eval("__import__('os').system('echo unsafe')")
except ValueError as e:
    print("Rejected:", e)

## 3. The tool loop

In [ ]:
for question in ["What is the weather in Paris?", "What is 12 * (3 + 4)?"]:
    out = run_tool_loop(get_llm(), registry, question)
    print("Q:", question)
    print("Trace:", out["trace"])
    print("A:", out["answer"], "\n")

Tool results go back into the conversation as `tool` messages, and the model writes the
final answer from them. The step limit stops a confused model from looping forever.
Try a question that needs no tool, and confirm the loop returns without calling one.

## Production checklist

- Validate every argument against the schema, and reject unexpected fields.
- Allowlist which tools each user or role may call.
- Give tools least-privilege credentials.
- Require human confirmation for actions with side effects (payments, deletions, sending messages).
- Time out and rate-limit tool calls.
- Return errors to the model in a structured form so it can recover.
- Log every call with arguments and results, minus sensitive data.
- Treat tool output as untrusted input. Web pages and documents can contain instructions aimed at the model (prompt injection).